# 02 — A global fixed-budget solution needs only polynomially many states.

Let a mask choose exactly $R$ of $N$ channels. There are $\binom NR$ such masks, but we do not need to visit them separately. Prefixes with the same numbers of selected and skipped channels share the same continuation problem.

This notebook derives a global oracle for the **fixed-$R$ importance/latency ratio**, recovers its mask, and compares it with Paper greedy, saturation tiles, and Top-$R$. Its Bellman subproblems are exact; a bounded search locates the optimal ratio to relative accuracy $10^{-6}$. The main inputs are Qwen activation vectors captured below, with their full projection widths. The cost coefficients come from Notebook 01.

The derivation, four selectors, and mask image follow the attached notebook. Paths below are relative to this notebook in `conclusion/python/results/02`. Set `RUN_ORACLE = True` to write the main comparison to `oracle_results_fixed_r.csv`; ordinary evaluation reads that CSV. Set `CAPTURE_TRACES = True` once to create the local NPZ from dense model forwards, then set it back to `False` to reuse the file. No switch runs an SSD benchmark. Existing CSVs are left intact. A new run recomputes the sweep; there is no automatic resume or input matching.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter1d
from itertools import product
from pathlib import Path
from IPython.display import Markdown, display
import json

MODEL = Path("../01/latency_model.json")
TRACE = Path("activation_traces.npz")
RESULTS = Path("oracle_results_fixed_r.csv")
UPSTREAM = Path("../../../upstream/vlm-flash")
PAPER_IMPL = "native"
CAPTURE_TRACES = False
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
PROMPTS = (
    "Explain why contiguous flash reads can be faster than scattered reads.",
    "Summarize the trade-off between neural-network sparsity and accuracy.",
    "Write a short Python function that computes a moving average.",
)
RUN_ORACLE = False
MAX_TRACES_PER_SHAPE = 32  # 0: all archived traces; no channel cropping.
BUDGETS = np.arange(1, 10) / 10

## Fix the objective before calling a solution global.

For $v_i\ge0$, let $\mathcal C(M)$ contain the **maximal** contiguous runs of ones in $M$. Define

$$I(M)=\sum_{i=0}^{N-1}v_iM_i,\qquad L(M)=\sum_{C\in\mathcal C(M)}T(|C|),\qquad \boxed{\eta_R^\star=\max_{\|M\|_1=R}\frac{I(M)}{L(M)}}.$$

The experiment fixes $R=\max(1,\lfloor fN\rfloor)$ from the requested fraction $f$, with $1\le R\le N$. Oracle, saturation tiles, and Top-$R$ select exactly $R$ rows; only Paper may select fewer. Its output never changes the oracle problem. Adjacent selected intervals form one run and are charged once. We assume the continuous two-line model exactly:

$$T(0)=0,\qquad T(\ell)=\begin{cases}a+b_1\ell,&1\le\ell\le s,\\b_2\ell,&\ell>s,\end{cases}\qquad a=(b_2-b_1)s,\quad 0\le b_1\le b_2,\quad b_2>0.$$

Notebook 01 supplies the fitted KiB model $T_{\rm KiB}(z)=c_1z+\delta\max(s_{\rm KiB},z)$ in milliseconds. For a weight row of $w$ KiB, substitute $z=w\ell$:

$$s=\frac{s_{\rm KiB}}w,\qquad b_1=wc_1,\qquad b_2=w(c_1+\delta).$$

The real-valued boundary $s$ is not rounded in the cost. This is a measured-and-fitted additive model, not a fresh timing of each mask. Runs outside the measured range use its assumed linear extension; native task splitting and scheduling are not represented here.

The earlier `Ratio exact` solved $1\le\|M\|_1\le R$, which can prefer a single short interval and leave budget unused. It is not the oracle for this fixed-cardinality comparison. The old notebook's matched-latency and matched-importance envelope gaps are replaced by a fixed-$R$ ratio gap. This is an explicit change of question, not a claim that the faster DP solves every coverage query or enumerates the entire Pareto frontier.

In [ ]:
model = json.loads(MODEL.read_text())
s_kib = model["s_kib"]
c1 = model["c1_ms_per_kib"]
delta = model["delta_ms_per_kib"]
s99 = model["s99_kib"]
display(Markdown(fr"$T_{{\rm KiB}}(z)={1000*c1:.6f}z+{1000*delta:.6f}\max({s_kib:g},z)$ µs."))

In [ ]:
def costs(n, s, b1, b2):
    length = np.arange(n + 1)
    T = b2 * length + (b2 - b1) * np.maximum(s - length, 0)
    T[0] = 0
    return T


def runs(mask):
    edges = np.flatnonzero(np.diff(np.r_[False, mask, False]))
    return edges.reshape(-1, 2)


def metrics(v, mask, T):
    spans = runs(mask)
    return v[mask].sum(), T[spans[:, 1] - spans[:, 0]].sum()

## Replace the ratio by a question with an additive answer.

For a proposed ratio $q\ge0$, define

$$F_R(q)=\max_{\|M\|_1=R}\{I(M)-qL(M)\}.$$

Because every feasible latency is positive,

$$\boxed{F_R(q)\ge0\iff q\le\eta_R^\star.}$$

Thus one additive optimization tells us whether $q$ is attainable. This is not a sweep intended to recover every Pareto point: we are locating the zero of the value function for one specified ratio objective.

## The last zero gives an exact recurrence.

Write $Z=N-R$ and $P_j=\sum_{i<j}v_i$. Let $D_{z,r}$ be the largest value of $I-qL$ over prefixes of length $z+r$ containing exactly $z$ zeros and $r$ ones. With no zeros there is only one mask, so

$$D_{0,r}=P_r-qT(r).$$

When $z>0$, place the **last zero** after $t$ selected channels. Before it there are $z-1$ zeros and $t$ ones; after it comes one run of $r-t$ ones. The empty final run $t=r$ is allowed. Therefore

$$\boxed{D_{z,r}=\max_{0\le t\le r}\left[D_{z-1,t}+P_{z+r}-P_{z+t}-qT(r-t)\right].}$$

**Why it is exact.** Every feasible prefix has this unique decomposition. For a fixed $t$, replacing its earlier prefix by the best $D_{z-1,t}$ cannot change the final run or its cost. Conversely, an optimal earlier prefix, one zero, and that final run construct a feasible prefix. Induction on $z$ proves equality, with $F_R(q)=D_{Z,R}$. The separating zero prevents charging two adjacent pieces as separate runs.

This recurrence is valid for any run-length table. There are only $(Z+1)(R+1)$ states, but directly scanning all $t$ still takes $O((Z+1)(R+1)^2)$ time. The two-line form removes that extra factor.

## Two range maxima replace the scan over run lengths.

For a fixed $z$, put

$$A_t=D_{z-1,t}-P_{z+t}+qb_1t,\qquad B_t=D_{z-1,t}-P_{z+t}+qb_2t.$$

Put $h=\lfloor s\rfloor$. Only the integer run-length window is rounded; the intercept remains $a=(b_2-b_1)s$. Separate a zero-length, short, and long final run:

$$D_{z,r}=\max\left\{\begin{aligned}&D_{z-1,r},\\&P_{z+r}-q(a+b_1r)+\max_{\max(0,r-h)\le t<r} A_t,\\&P_{z+r}-qb_2r+\max_{0\le t<r-h} B_t.\end{aligned}\right.$$

An empty maximum is $-\infty$. The short range is a moving window; the long range grows by one index at a time. Both maxima for an entire row can be computed in $O(R+1)$ time. A moving maximum maintains only undominated indices: each enters and leaves its queue at most once.

Below, `trailing_max(a, w)[j]` means $\max_{\max(0,j-w+1)\le t\le j}a_t$. The SciPy routine implements that linear-time moving maximum; its `origin` places the window behind the current index. `np.maximum.accumulate` is the ordinary prefix maximum. There is no restricted corridor or pruned mask family.

In [ ]:
def trailing_max(values, width):
    return maximum_filter1d(values, size=width, origin=(width - 1) // 2,
                            mode="constant", cval=-np.inf)

In [ ]:
def bellman(v, R, q, s, b1, b2):
    Z = len(v) - R
    P = np.r_[0., np.cumsum(v)]
    r = np.arange(R + 1)
    D = np.empty((Z + 1, R + 1))
    D[0] = P[:R + 1] - q * costs(R, s, b1, b2)
    width = min(int(np.floor(s)), R)
    for z in range(1, Z + 1):
        prefix = P[z:z + R + 1]
        previous = D[z - 1] - prefix
        short = np.full(R + 1, -np.inf)
        if width:
            short[1:] = trailing_max(previous[:-1] + q * b1 * r[:-1], width)
        long = np.full(R + 1, -np.inf)
        long[width + 1:] = np.maximum.accumulate(previous + q * b2 * r)[:R - width]
        D[z] = np.maximum.reduce([
            D[z - 1],
            prefix - q * ((b2 - b1) * s + b1 * r) + short,
            prefix - q * b2 * r + long])
    return D

### Recover a mask, not just its objective value.

At $(z,r)$, choose a maximizing $t$ in the original recurrence. Mark $[z+t,z+r)$ selected and continue at $(z-1,t)$. At $z=0$, the remaining prefix consists entirely of ones. Ties can give different optimal masks.

We keep the value table and rescan one row per backtracking step rather than maintaining another table of predecessors. This also costs $O((Z+1)(R+1))$. Including recovery, one additive solve uses

$$\boxed{O((N-R+1)(R+1))\text{ time and memory}.}$$

The memory bound includes the saved table; it is not a claim of rolling-row memory with free traceback. For $R\simeq N/2$, both bounds are quadratic, not exponential. The extra ones in the formula cover $R=N$ correctly.

In [ ]:
def recover(v, R, q, T, D):
    P = np.r_[0., np.cumsum(v)]
    mask = np.zeros(len(v), bool)
    r = R
    for z in range(len(v) - R, 0, -1):
        t = np.arange(r + 1)
        scores = D[z - 1, :r + 1] + P[z + r] - P[z + t] - q * T[r - t]
        start = int(np.argmax(scores))
        mask[z + start:z + r] = True
        r = start
    mask[:r] = True
    return mask

## Paper greedy keeps the reference implementation's budget rule.

**Paper greedy** calls the checked-out [reference selector](../../../upstream/vlm-flash/src/vlmflash/policy.py), rather than reproducing its candidate loop. The window start, step, and stride cap follow the shape-specific settings used in the earlier CSV experiment. The candidate ceiling uses the separate $s_{99}$ throughput crossing; candidate costs come from Notebook 01's fitted model. This is the reference algorithm retargeted to that model, not a reproduction of the paper's device timings.

Whole windows can leave the requested budget $R$ partly unfilled: $R_P=|M_P|\le R$. We neither pad Paper's mask nor reduce the other methods' budgets. Paper is a comparison baseline, not the oracle's initial mask. Final costs are recomputed on maximal runs by `metrics`, not taken from the selector's sum over accepted windows.

In [ ]:
PAPER_GEOMETRY = {"896x128": (8, 8), "896x896": (8, 8),
                  "896x4864": (8, 8), "4864x896": (12, 16)}


def load_paper():
    import sys, torch
    sys.path.insert(0, str((UPSTREAM / "src").resolve()))
    import vlmflash
    assert Path(vlmflash.__file__).resolve().is_relative_to(UPSTREAM.resolve())
    table = vlmflash.LatencyTable({
        z: c1*z + delta*max(s_kib, z)
        for z in range(1, int(np.ceil(max(s_kib, s99))) + 2)})

    def select(v, R, w, shape):
        start, jump = PAPER_GEOMETRY[shape]
        params = vlmflash.ChunkParams(start_kb=start, step_kb=start,
                                     jump_cap_kb=jump, end_kb=s99 + 1)
        result = vlmflash.select_chunks(torch.tensor(v, dtype=torch.float32),
            R, w, table, params, impl=PAPER_IMPL)
        return result.mask.cpu().numpy()

    return select

## Start from a contiguous run, independently of Paper.

Let $M_C$ be the most important contiguous interval of exactly $R$ rows. Prefix sums find it directly. For the two-line model, merging all selected rows into one run never increases cost. Indeed, if the positive run lengths sum to $R$, then

$$L(M)=b_1R+(b_2-b_1)\sum_C\max(s,|C|)\ge b_1R+(b_2-b_1)\max(s,R)=T(R).$$

The contiguous seed attains this minimum latency, while Top-$R$ gives the largest possible importance. Thus

$$\boxed{\ell_0=\frac{I(M_C)}{T(R)}\le\eta_R^\star\le u_0=\frac{I(\mathrm{Top}\text{-}R)}{T(R)}.}$$

The lower bound belongs to a real mask; the upper bound combines the best numerator and denominator, which need not belong to the same mask. This does not claim that the contiguous seed maximizes importance or the ratio. It is feasible even when Paper underfills or returns an empty mask. For $R\ge s$, $T(R)=b_2R$; for $R<s$, $T(R)$ is a stronger denominator bound.

Test $q=(\ell+u)/2$. If $F_R(q)<0$, replace $u$ by $q$. Otherwise recover a feasible mask and replace $\ell$ by its ratio. Each step at least halves the bracket. Stop when $u-\ell\le\epsilon\ell_0$; the returned mask satisfies

$$0\le1-\frac{I(M)/L(M)}{\eta_R^\star}\le\epsilon.$$

For nonzero importance, some length-$R$ interval contains a positive entry, so $\ell_0>0$. Put $\kappa_0=u_0/\ell_0$; at most $\lceil\log_2(\kappa_0/\epsilon)\rceil$ additive solves suffice. Zero importance gives two zero bounds immediately.

After constructing the contiguous seed and the Top-$R$ importance bound, the search costs $O((N-R+1)(R+1)\log(\kappa_0/\epsilon))$. The oracle has no Paper input or selector dependency. The recurrence is exact mathematically; the numerical bounds use float64, not formal interval arithmetic.

In [ ]:
def top_r(v, R):
    mask = np.zeros(len(v), bool)
    mask[np.argsort(-v, kind="stable")[:R]] = True
    return mask

In [ ]:
def solve(v, R, s, b1, b2, rtol=1e-6):
    assert 1 <= R <= len(v) and s > 0 and 0 <= b1 <= b2 and b2 > 0
    assert 0 < rtol < 1 and np.isfinite(v).all() and (v >= 0).all()
    T = costs(len(v), s, b1, b2)
    P = np.r_[0., np.cumsum(v)]
    start = np.argmax(P[R:] - P[:-R])
    mask = np.zeros(len(v), bool)
    mask[start:start + R] = True
    I, L = metrics(v, mask, T)
    lower = I / L
    upper = v[top_r(v, R)].sum() / T[R]
    assert lower > 0 or upper == 0
    tolerance = rtol * lower
    calls = 0
    while upper - lower > tolerance:
        q = (lower + upper) / 2
        assert lower < q < upper, "Tolerance is below float64 resolution."
        D = bellman(v, R, q, s, b1, b2)
        calls += 1
        if D[-1, -1] >= 0:
            mask = recover(v, R, q, T, D)
            I, L = metrics(v, mask, T)
            lower = I / L
        else:
            upper = q
    return mask, lower, upper, calls

## Enumeration is a check of the recurrence, not the main oracle.

On at most nine channels we enumerate all masks independently. A separate column-by-column run counter evaluates their costs. We check every positive row budget, several ratio candidates, and the final bracket. The cases include zero importance, zero short slope, equal slopes, fractional and sub-row boundaries, $s=1$, and $s>N$. Comparing values rather than masks allows legitimate ties.

In [ ]:
def enumerate_small(v, T):
    masks = np.array(list(product([False, True], repeat=len(v))))
    latency = np.zeros(len(masks))
    length = np.zeros(len(masks), int)
    for column in masks.T:
        latency[~column] += T[length[~column]]
        length = np.where(column, length + 1, 0)
    return masks, masks @ v, latency + T[length]

In [ ]:
checks = 0
for n in (1, 5, 9):
    for v in (np.zeros(n), np.random.default_rng(n).uniform(0, 4, n)):
        for boundary in (0.5, 1, 2.5, 4, n + 2):
            for short_slope in (0., 0.25, 1.):
                T = costs(n, boundary, short_slope, 1.)
                masks, imp, delay = enumerate_small(v, T)
                for R in range(1, n + 1):
                    allowed = masks.sum(axis=1) == R
                    for q in (0., 0.1, 1., 10.):
                        D = bellman(v, R, q, boundary, short_slope, 1.)
                        mask = recover(v, R, q, T, D)
                        I, L = metrics(v, mask, T)
                        exact = np.max(imp[allowed] - q * delay[allowed])
                        assert mask.sum() == R
                        np.testing.assert_allclose([D[-1, -1], I - q * L], exact, atol=1e-9)
                        checks += 1
                    mask, lower, upper, calls = solve(v, R, boundary, short_slope, 1.)
                    np.testing.assert_allclose(delay[allowed].min(), T[R])
                    assert mask.sum() == R
                    exact = np.max(imp[allowed] / delay[allowed])
                    assert lower - 1e-9 <= exact <= upper + 1e-9
                    np.testing.assert_allclose(metrics(v, mask, T)[0] / metrics(v, mask, T)[1], lower)
                    assert upper - lower <= 1e-6 * lower + 1e-9
pd.Series({"Additive values and recovered masks checked": checks,
           "Largest enumerated channel count": 9})

**Saturation tiles** restrict the geometry instead. For each grid offset $0$ and $\lfloor B/2\rfloor$, choose the $\lfloor R/B\rfloor$ highest-importance full tiles, then the best disjoint interval of length $R\bmod B$. Choose the better resulting mask by its merged-run ratio. For $R<B$, choose the best length-$R$ interval. Top-$R$ simply selects the largest individual values. Oracle, tiles, and Top-$R$ have exactly the requested $R$ rows; Paper may have fewer. All are evaluated by `metrics`, which merges adjacent pieces before charging latency.

Here $B=\lceil s\rceil=\lceil s_{\rm KiB}/w\rceil$ is the integer tile width passed as `s` to the unchanged function below. It is tied to the fitted constant-cost-per-byte boundary, not a hard-coded 240 KiB or the separate 99% throughput crossing. For $B>N$, every feasible budget uses the best single interval.

In [ ]:
def tiles(v, R, T, s):
    P = np.r_[0., np.cumsum(v)]
    if R < s:
        start = np.argmax(P[R:] - P[:-R])
        mask = np.zeros(len(v), bool)
        mask[start:start + R] = True
        return mask
    choices = []
    k, remaining = divmod(R, s)
    for offset in (0, s // 2):
        starts = np.arange(offset, len(v) - s + 1, s)
        if len(starts) < k:
            continue
        weights = P[starts + s] - P[starts]
        mask = np.zeros(len(v), bool)
        for start in starts[np.argsort(-weights, kind="stable")[:k]]:
            mask[start:start + s] = True
        if remaining:
            occupied = np.r_[0, np.cumsum(mask)]
            valid = occupied[remaining:] == occupied[:-remaining]
            if not valid.any():
                continue
            scores = np.where(valid, P[remaining:] - P[:-remaining], -np.inf)
            start = np.argmax(scores)
            mask[start:start + remaining] = True
        I, L = metrics(v, mask, T)
        choices.append((I / L, mask))
    return max(choices, key=lambda row: row[0])[1]

## Capture the projection inputs here, then read the local NPZ.

Set `CAPTURE_TRACES = True` to load the declared Qwen checkpoint and run one dense forward per prompt. The first capture requires PyTorch and Transformers; an uncached checkpoint is downloaded. Later runs read `activation_traces.npz` in this directory with `CAPTURE_TRACES = False`. A missing file raises an error on a requested comparison rather than starting a model download. Reading the saved CSV does not require a capture or the selector dependencies.

A pre-forward hook records $v_i=\operatorname{mean}_{\rm batch,token}|x_i|$ for each attention or MLP projection. It returns nothing and does not change the input or dense output. The reduction follows the [reference importance expression](../../../upstream/vlm-flash/src/vlmflash/linear.py): take the absolute value and mean in the activation dtype, then convert the result to float32. The NPZ contains those vectors, projection dimensions, prompt indices, model revision and dtype; it is not a weight archive.

Select up to 32 evenly spaced traces per shape by default, or all traces with a zero cap. No channels are cropped, sorted, or tiled before selection. Normalize each vector to mean one. The budgets $R/N\simeq0.1,\ldots,0.9$ define the trajectory; many layers and budgets from three prompts are not independent prompts.

In [ ]:
def capture_traces():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    net = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.bfloat16).to(device).eval()
    entries, arrays, hooks = [], {}, []

    def record(name):
        def hook(layer, inputs):
            n, d = layer.in_features, layer.out_features
            key = f"values_{len(entries):06d}"
            arrays[key] = inputs[0].detach().abs().reshape(-1, n).mean(0).float().cpu().numpy()
            entries.append(dict(trace_id=len(entries), input_index=input_index,
                module=name, n=n, d=d, shape=f"{n}x{d}", array=key,
                num_tokens=int(tokens["input_ids"].numel())))
        return hook

    try:
        for name, layer in net.named_modules():
            if isinstance(layer, torch.nn.Linear) and ".layers." in name and name.endswith("_proj"):
                hooks.append(layer.register_forward_pre_hook(record(name)))
        with torch.inference_mode():
            for input_index, prompt in enumerate(PROMPTS):
                text = tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
                tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
                net(**tokens, use_cache=False)
                print(f"{input_index + 1}/{len(PROMPTS)} prompts captured", end="\r")
    finally:
        for handle in hooks:
            handle.remove()
    assert entries, "No projection inputs were captured."
    manifest = dict(model=MODEL_ID, model_revision=MODEL_REVISION,
        model_dtype=str(next(net.parameters()).dtype).split(".")[-1],
        prompts=list(PROMPTS), prompt_count=len(PROMPTS), traces=entries, trace_count=len(entries))
    np.savez_compressed(TRACE, manifest_json=np.asarray(json.dumps(manifest)), **arrays)

In [ ]:
traces = []
if CAPTURE_TRACES:
    capture_traces()
if RUN_ORACLE:
    with np.load(TRACE, allow_pickle=False) as archive:
        manifest = json.loads(str(archive["manifest_json"].item()))
        traces = [{**t, "values": archive[t["array"]].astype(float)} for t in manifest["traces"]]
    if MAX_TRACES_PER_SHAPE:
        groups = [[t for t in traces if t["shape"] == shape] for shape in sorted({t["shape"] for t in traces})]
        traces = [g[i] for g in groups for i in np.linspace(
            0, len(g) - 1, min(len(g), MAX_TRACES_PER_SHAPE), dtype=int)]
    traces.sort(key=lambda t: t["trace_id"])
    weight_bytes = {"float16": 2, "bfloat16": 2, "float32": 4}[manifest["model_dtype"]]
    shapes = pd.DataFrame(traces).groupby(["shape", "n", "d"]).size().rename("traces").reset_index()
    shapes["KiB/row"] = weight_bytes * shapes.d / 1024
    shapes["tile rows"] = np.ceil(s_kib / shapes["KiB/row"]).astype(int)
    display(shapes)

For a heuristic mask with **exactly the requested $R$ rows** and ratio $u_M=I(M)/L(M)$, the true gap is $g(M)=100(1-u_M/\eta_R^\star)$. The numerical bracket yields

$$\max(0,100(1-u_M/\ell))\le g(M)\le\max(0,100(1-u_M/u)).$$

We show the upper end of this interval, rather than silently treating the oracle's finite tolerance as zero. Its uncertainty is at most $0.0001$ percentage points in exact arithmetic. Importance, latency, and run count are retained separately; a larger ratio does not mean larger importance.

An underfilled Paper mask is outside this fixed-$R$ feasible set. Its gap is left undefined, not clipped to zero: its ratio can even exceed $\eta_R^\star$ without contradicting optimality. Its importance, latency, fill percentage, and fragmentation remain visible. Paper receives a fixed-$R$ gap only in cases where it fills $R$ exactly.

## The CSV preserves the comparison, including unfilled budgets.

Write `oracle_results_fixed_r.csv` after each completed trace; a new explicit run starts the sweep again. With `RUN_ORACLE = False`, all result figures are read from that file. Earlier matched-cardinality CSVs are different experiments and cannot be converted by relabelling their rows. They remain untouched. After changing inputs or settings, recompute the results or choose another `RESULTS` path.

`R_nominal` is the requested $R$, fixed once per case. `R_selected` is $|M|$ for each method. The fill percentage is $100|M|/R$. Fragmentation is recorded both as the number of maximal runs and as the short-run deficit,

$$D(M)=\sum_{C\in\mathcal C(M)}(s-|C|)_+,\qquad L(M)-b_2|M|=(b_2-b_1)D(M).$$

Using $|M|$, not the requested $R$, makes the identity valid for Paper too. Empty Paper selections and zero-importance inputs retain their rows and status; they have no ratio gap. An empty Paper mask does not prevent the other three methods from solving the requested problem. Saved run intervals reconstruct the mask images without solving the oracle again.

In [ ]:
if RUN_ORACLE:
    paper = load_paper()
    records = []
    for i, trace in enumerate(traces, 1):
        v = trace["values"]
        v = (v / v.mean()).astype(np.float32).astype(float) if v.any() else v
        N = len(v)
        w = weight_bytes * trace["d"] / 1024
        s, b1, b2 = s_kib / w, w * c1, w * (c1 + delta)
        T = costs(N, s, b1, b2)
        for fraction in BUDGETS:
            R = max(1, int(fraction * N))
            oracle, lower, upper, calls = solve(v, R, s, b1, b2)
            methods = {"Global DP": oracle, "Paper greedy": paper(v, R, w, trace["shape"]),
                       "Saturation tiles": tiles(v, R, T, int(np.ceil(s))), "Top-R": top_r(v, R)}
            for name, mask in methods.items():
                selected = int(mask.sum())
                assert 0 <= selected <= R if name == "Paper greedy" else selected == R
                status = "empty_selection" if selected == 0 else "ok" if v.any() else "zero_importance"
                I, L = metrics(v, mask, T)
                spans = runs(mask)
                length = np.diff(spans, axis=1).ravel()
                shortfall = np.maximum(s - length, 0).sum()
                np.testing.assert_allclose(L - b2*selected, (b2-b1)*shortfall, atol=1e-10)
                comparable = selected == R and lower > 0
                if comparable:
                    assert I / L <= upper + 1e-8 * max(1, upper)
                records.append(dict(trace_id=trace["trace_id"], shape=trace["shape"],
                    prompt=trace["input_index"], fraction=fraction, N=N, R_nominal=R,
                    R_selected=selected, fill_pct=100*selected/R, method=name, status=status, importance=I,
                    latency=L, retained_pct=100*I/v.sum() if v.any() else np.nan,
                    chunks=len(spans), short_chunks=int((length < s).sum()),
                    shortfall_rows=shortfall, overhead_ms=(b2-b1)*shortfall,
                    gap_lower_pct=max(0, 100*(1-I/L/lower)) if comparable else np.nan,
                    gap_upper_pct=max(0, 100*(1-I/L/upper)) if comparable else np.nan,
                    oracle_calls=calls, intervals=json.dumps(spans.tolist())))
        pd.DataFrame(records).to_csv(RESULTS, index=False)
        print(f"{i}/{len(traces)} traces saved", end="\r")
results = pd.read_csv(RESULTS) if RESULTS.is_file() else pd.DataFrame()
assert results.empty or "R_selected" in results, "Recompute the fixed-R CSV; old results used a different budget."
frame = results[results.status == "ok"].copy() if not results.empty else results

In [ ]:
def saved_mask(row):
    mask = np.zeros(int(row.N), bool)
    for start, end in json.loads(row.intervals):
        mask[start:end] = True
    return mask


examples = {}
if not results.empty:
    for trace_id, group in results[np.isclose(results.fraction, .5)].groupby("trace_id"):
        examples[trace_id] = {row.method: saved_mask(row) for row in group.itertuples()}

In [ ]:
if not frame.empty:
    eligible = frame[(frame.R_selected == frame.R_nominal) & frame.gap_upper_pct.notna()]
    gaps = eligible[eligible.method != "Global DP"].groupby(["shape", "method"]).gap_upper_pct.median().unstack()
    ax = gaps.plot.bar(figsize=(8, 3.5), rot=0)
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    plt.ylim(0, 105)
    plt.xlabel("Projection shape")
    plt.ylabel("Median ratio-gap upper bound (%)")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

**Figure 1.** Median gaps over the configured Qwen traces and row budgets, grouped by projection shape. These are comparisons under the same fitted cost, not measured improvements in SSD speed. Paper now uses the reference implementation; the other three methods retain their definitions. Only masks selecting exactly the requested $R$ enter this gap chart. Paper is therefore summarized on its fully filled cases only; the table below reports the eligible count and total for each method. Underfill is shown separately, not assigned a zero gap.

Tiles can benefit from coherent importance while fixed boundaries may miss valuable rows. The exact fixed-budget problem supplies the comparison standard, rather than a favorable example. This chart alone does not establish a relation between retained activation magnitude and end-to-end model quality.

In [ ]:
if not results.empty:
    paper_rows = results[results.method == "Paper greedy"]
    display(paper_rows.groupby(["shape", "fraction"]).fill_pct.agg(["min", "median", "count"]))
    display(paper_rows.groupby("status").size().rename("cases"))
    display(results.groupby(["shape", "method"]).gap_upper_pct.agg(["count", "size"]).rename(
        columns={"count": "gap eligible", "size": "total cases"}))
if not frame.empty:
    display(frame.groupby(["shape", "fraction", "method"])[
        ["chunks", "short_chunks", "shortfall_rows", "overhead_ms"]].median().round(4))

In [ ]:
if examples:
    example_id = min(examples)
    example = results[(results.trace_id == example_id) & np.isclose(results.fraction, 0.5)]
    display(example.set_index("method")[["R_nominal", "R_selected", "importance", "latency", "chunks", "gap_upper_pct"]].round(3))

In [ ]:
if not frame.empty:
    for shape, data in frame.groupby("shape", sort=False):
        points = data[data.trace_id == data.trace_id.min()]
        plt.figure(figsize=(8, 3.5))
        for name, group in points.groupby("method", sort=False):
            group = group.sort_values("fraction")
            plt.plot(group.latency, group.retained_pct, "o-", label=name, ms=4)
            if name == "Global DP":
                for row in group.itertuples():
                    plt.annotate(str(row.R_nominal), (row.latency, row.retained_pct), fontsize=7)
        plt.xscale("log")
        plt.xlabel("Model latency L (ms; log scale)")
        plt.ylabel("Retained importance (%)")
        plt.title(f"{shape}: trace {points.trace_id.iloc[0]}; labels are requested R")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

**Figure 2.** Each point is an actual mask at one requested row budget; connect points in budget order. Each shape uses its first selected trace so the four trajectories describe the same importance vector, rather than independently averaged coordinates. Oracle labels give the requested $R$, also used exactly by tiles and Top-$R$. Paper may retain fewer rows at the same requested budget. These are operational trade-offs, not same-cardinality dominance claims. Empty masks have zero latency and are omitted from the log plot; the CSV and mask images retain them.

The horizontal scale is logarithmic. The DP points are fixed-$R$ ratio optima to tolerance, **not a complete Pareto frontier**. A larger retained importance can cost more latency; the ratio trades these quantities rather than maximizing each independently. Lines guide the eye and do not imply that intermediate points are feasible masks.

In [ ]:
if examples:
    methods = examples[example_id]
    N = len(methods["Global DP"])
    R = int(methods["Global DP"].sum())
    plt.figure(figsize=(9, 2.4))
    plt.imshow(np.array(list(methods.values())), aspect="auto", interpolation="nearest")
    plt.yticks(range(len(methods)), [f"{name} (selected={int(mask.sum())})" for name, mask in methods.items()])
    plt.xticks(np.linspace(0, N, 8, dtype=int))
    plt.xlabel(f"Channel (trace {example_id}; requested R = {R}; Paper may underfill)")
    plt.tight_layout()
    plt.show()

**Figure 3.** Bright cells are selected. As in the attachment, the four masks are shown together for the same input and requested half-row budget. Each row label gives its actual selected count; only Paper may underfill. No mask is replaced by its contiguity histogram. The model can also be written

$$L(M)=b_2|M|+(b_2-b_1)\sum_{C\in\mathcal C(M)}(s-|C|)_+.$$

Only short runs incur the extra penalty. This does **not** prove that every optimum uses only long runs: a valuable short run can be worth its penalty. The DP does not impose the tile heuristic's geometry.

### Trace the non-monotone latency point.

For the first `896x128` trace, reconstruct every saved mask at nominal 70%, 80%, and 90%, when available in the CSV. Each row label reports selected/requested rows, number of merged runs, and fitted latency. This exposes whether a backward step on Figure 2 comes from a change in mask fragmentation.

In [ ]:
JUMP_SHAPE = "896x128"
JUMP_FRACTIONS = (0.7, 0.8, 0.9)

if not results.empty and JUMP_SHAPE in set(results["shape"]):
    jump_trace_id = int(results.loc[results["shape"] == JUMP_SHAPE, "trace_id"].min())
    points = results[(results.trace_id == jump_trace_id) & results.fraction.isin(JUMP_FRACTIONS)]
    jump_rows, jump_labels = [], []
    for row in points.sort_values("fraction", kind="stable").itertuples():
        jump_rows.append(saved_mask(row))
        jump_labels.append(
            f"{row.fraction:.0%} · {row.method} | selected/R={row.R_selected}/{row.R_nominal}, chunks={row.chunks}, L={row.latency:.6f} ms")
    if jump_rows:
        N = int(points.N.iloc[0])
        fig, ax = plt.subplots(figsize=(12, 5.2))
        ax.imshow(np.asarray(jump_rows), aspect="auto", interpolation="nearest")
        ax.set_yticks(range(len(jump_rows)), jump_labels, fontsize=8)
        ax.set_xticks(np.linspace(0, N, 8, dtype=int))
        for boundary in range(4, len(jump_rows), 4):
            ax.axhline(boundary - 0.5, color="white", linewidth=1.5)
        ax.set_xlabel(f"Channel ({JUMP_SHAPE}, trace {jump_trace_id}); bright cells are selected")
        ax.set_title("Mask geometry around the latency reversal")
        plt.tight_layout()
        plt.show()

### What is established

The oracle searches the whole fixed-cardinality mask space at the requested $R$ under the specified two-line cost, independently of Paper's fill count. Only Paper is allowed to underfill; its cross-cardinality trade-off is not a fixed-$R$ optimality gap. Its additive steps are exact, and its ratio bracket quantifies the remaining numerical optimization error. The recovered importance and latency belong to an actual mask, not an interpolated point.

This is an offline model-based reference, not a claim of a two-millisecond online selector or a measured SSD speedup. Replacing the analytical cost by an arbitrary lookup table preserves the last-zero recurrence but not its two-range acceleration. A coverage optimum or an entire Pareto frontier is a different problem; it is not supplied merely by sweeping a scalar price.

The old small-mask illustration is now a verification step. State compression, rather than a limit on channel count, supplies the global comparison.

### Mathematical context

The fixed-budget objective follows [idea.md](../../../../preliminary_research/idea.md). The selected/skipped-count formulation extends the full-state oracle in [Experiment 28](../../../../experiments/28_exact_corridor/README.md); unlike a fixed count of Dinkelbach updates, the bracket above bounds ratio error and iteration count. [Experiment 07](../../../../experiments/07_exact_n4864/README.md) used a different, affine-cost coverage DP. SciPy documents the moving maximum's linear-time bound in [maximum_filter1d](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.maximum_filter1d.html). These links are context, not runtime inputs.